In [13]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
load_dotenv()

True

In [4]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [9]:
llm = ChatOpenAI()
def chat_node(state:ChatState):
    # take user query from State
    messages = state['messages']


    # send to llm 
    response = llm.invoke(messages)
    # response store state

    return {'messages':[response]}

In [14]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

work_flow = graph.compile(checkpointer=checkpointer)

In [11]:
initial_state = {
    'messages':[HumanMessage(content="What is the capital of India")]
}
work_flow.invoke(initial_state)

{'messages': [HumanMessage(content='What is the capital of India', additional_kwargs={}, response_metadata={}, id='ff2b1cb1-32a7-4f1b-b2e6-66a5f6410405'),
  AIMessage(content='The capital of India is New Delhi.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 13, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EAwO2FxXClqEsTX83d8210zBSgreB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe65f-2f4f-7373-9f4f-b155b194688e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 8, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'c

In [15]:
thread_id ='1'
while True:
    user_message = input("Type Here")
    print("User:",user_message)
    if user_message.strip().lower() in ['bye','exit','quit']:
        break
    config = {'configurable':{'thread_id':thread_id}}
    response = work_flow.invoke({'messages':HumanMessage(content=user_message)},config=config)

    print("AI",response['messages'][-1].content)

User: HI my name saurav
AI Hello Saurav! How can I assist you today?
User: what is my name
AI Your name is Saurav.
User: bye
